<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#14213d 0%,#2a4a9b 50%,#3f7fd9 80%,#b22222 100%);padding:30px 32px;border-radius:14px;color:#ffffff;"><div style="font-size:12.5px;letter-spacing:2.5px;text-transform:uppercase;color:#ffffff;">Santa 2026 &middot; The Elf Calculator</div><div style="font-size:34px;font-weight:800;line-height:1.15;margin-top:8px;color:#ffffff;">📈 Best Score + What Changed Today</div><div style="font-size:17px;line-height:1.5;margin-top:10px;color:#ffffff;">The best public programs for all 1,000 puzzles, ready to submit, and a daily digest of the last 24 hours: which puzzles got shorter, by how much, who found it, and which public releases set a new best score. Every number is computed in this notebook from the attached files. </div></div>

**Score of this version: 128,291 instructions.**

**Use it:** Copy & Edit → set `WRITE_SUBMISSION = True` in the first code cell → Run All. About 30 seconds, every one of
the 1,000 programs is replayed before `submission.csv` is written.

* **Input:** only the dataset [santa-2026-elf-programs-and-experiments](https://www.kaggle.com/datasets/asteriospanoras/santa-2026-elf-programs-and-experiments). Its `best_programs.csv`
  is the base: for every puzzle the shortest public program that replays correctly, with its author in the `source`
  column (full credits and history are in the dataset).
* **This version's change** is in the `CHANGE` cell. It is replayed like every other program before anything is written.

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#eaf6f0;border-left:6px solid #0f9a68;border-radius:8px;padding:12px 18px;margin:10px 0;"><b>What you will find.</b><ol style="margin:6px 0 4px 0;padding-left:22px;color:#111111;"><li><b>What changed today</b>: instructions saved against the best public file of about a day ago, every improved puzzle, the gain by contributor and by program length, and the new best public scores of the day.</li><li><b>Where the gap sits</b>: by program length, number of targets and target size.</li><li><b>Where everyone still loses</b>: a proven lower bound per puzzle and how many instructions above it today's best programs spend.</li><li><b>Your submission</b>: set <code style="background:#e3e8f2;color:#111111;padding:1px 6px;border-radius:4px;">MY_SUBMISSION</code> to your file and the same audit runs on it, with a strict checker that names the instruction and the rule it breaks.</li></ol></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15px;color:#111111;background:#ffffff;padding:4px 6px;border-radius:6px;"><b>How to read it.</b> Charts and short explanations come first; the analysis code cells are collapsed (click to open). Contents: <a href="#p1">1 what changed today</a> &middot; <a href="#p2">2 where the gap sits</a> &middot; <a href="#p3">3 where everyone loses</a> &middot; <a href="#p4">4 your submission</a>. The submission part of the notebook (rules, base, change, write) is unchanged and visible.</div>


In [ ]:
WRITE_SUBMISSION = False  # set True after Copy & Edit to write /kaggle/working/submission.csv


## Rules as code

In [ ]:
import csv, glob, json, os
csv.field_size_limit(1 << 30)
DATASET = 'santa-2026-elf-programs-and-experiments'
def find(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True), key=lambda h: (DATASET not in h, h))
    if not hits: raise FileNotFoundError(f'{name}: attach the dataset {DATASET}')
    return hits[0]

def load_puzzles(path):
    return {r['id']: dict(start=json.loads(r['starting_values']), targets=json.loads(r['targets']),
                          limit=int(r['value_limit'])) for r in csv.DictReader(open(path))}

def replay(p, prog):
    """Strict replay under the competition rules; returns the length or raises ValueError.
    48 registers, read registers must hold a value, values stay positive integers <= value_limit,
    subtraction needs a > b, division must be exact, every target must appear in some register."""
    R = [None] * 48
    for k, v in enumerate(p['start']): R[k] = v
    seen = set(p['start'])
    for op, i, j, d in prog:
        a, b = R[i], R[j]
        if op not in '+-*/' or a is None or b is None: raise ValueError('bad instruction')
        if op == '-' and a <= b: raise ValueError('non-positive subtraction')
        if op == '/' and a % b: raise ValueError('inexact division')
        v = a + b if op == '+' else a - b if op == '-' else a * b if op == '*' else a // b
        if not 0 < v <= p['limit']: raise ValueError('value out of range')
        R[d] = v; seen.add(v)
    if not set(p['targets']) <= seen: raise ValueError('missing target')
    return len(prog)

## Base: the dataset's best public programs

In [ ]:
P = load_puzzles(find('puzzles.csv'))
programs = {r['id']: json.loads(r['program']) for r in csv.DictReader(open(find('best_programs.csv')))}
BASE = sum(replay(P[k], programs[k]) for k in P)
print(f'base from the dataset: {BASE:,} instructions, all 1,000 programs replay correctly')

## 🎁 Behind today's door: the one change of this version

In [ ]:
CHANGE = {}   # no change in this version: best_programs.csv already holds the best public program of every puzzle

for pid, prog in CHANGE.items():
    old, new = replay(P[pid], programs[pid]), replay(P[pid], prog)   # the new program is checked like every other one
    assert new < old, (pid, old, new)
    programs[pid] = prog
    print(f'{pid}: {old} -> {new}')
TOTAL = sum(replay(P[k], programs[k]) for k in P)
print(f'total after the change: {TOTAL:,} instructions ({TOTAL - BASE:+d})')

<a id="p1"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#14213d,#2a6fdb);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 1</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">What changed today</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">Puzzles that got shorter in the last ~24 hours of public progress, and who made them shorter</div></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#f3f6fc;border-left:6px solid #2a6fdb;border-radius:8px;padding:12px 18px;margin:10px 0;">Every number is a per-puzzle comparison: for each of the 1,000 puzzles, the length of the best public program of about a day ago (<code style="background:#e3e8f2;color:#111111;padding:1px 6px;border-radius:4px;">public_day_ago.csv</code>) minus the length of today's best program (the one this notebook submits). Both are replayed under the competition rules first. The dataset refreshes the day-old file at every release, so this page always shows the last day of public progress. The list of new best scores comes from <code style="background:#e3e8f2;color:#111111;padding:1px 6px;border-radius:4px;">public_lineage.csv</code>.</div>

In [ ]:
# Analysis toolkit: strict checker, per-puzzle gap table, figures and cards. Reads only the attached files.
import math, time, traceback
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
T0 = time.time()
# One figure style for Kaggle light AND dark pages: solid white figures with dark text.
INK, MUTED, GRID = '#111111', '#555b66', '#dfe3ea'
NAVY, BLUE, RED, GOLD, GREEN, PURPLE, GREY = '#14213d', '#2a6fdb', '#c0392b', '#c98a00', '#0f9a68', '#6a4c93', '#9aa3b2'
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'figure.dpi': 110, 'font.size': 11.5, 'axes.titlesize': 13.5, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.labelsize': 11.5, 'axes.edgecolor': '#333333', 'axes.labelcolor': INK, 'xtick.color': INK, 'ytick.color': INK,
    'text.color': INK, 'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
FONT = "font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;"

def run(fn, *a):
    """Run one analysis step; an error here is printed and never stops the notebook (the submission part must finish)."""
    try: return fn(*a)
    except Exception: print('analysis step skipped:'); traceback.print_exc(limit=2)

# ---------- strict checker ----------
OPS = ('+', '-', '*', '/')
def explain(p, prog):
    """None if the program is valid under the competition rules, else the first problem in plain words.
    Stricter than replay(): it also rejects operators like '+-', negative or non-integer registers, booleans."""
    if not isinstance(prog, list): return f'the program is not a JSON list (got {type(prog).__name__})'
    R = [None] * 48
    for k, v in enumerate(p['start']): R[k] = v
    seen = set(p['start'])
    for n, ins in enumerate(prog):
        at = f'instruction {n} {json.dumps(ins)}'
        if not isinstance(ins, list) or len(ins) != 4: return f'{at}: needs 4 fields [op, i, j, d]'
        op, i, j, d = ins
        if not isinstance(op, str) or op not in OPS: return f'{at}: operator must be one of + - * /'
        for name, r in (('i', i), ('j', j), ('d', d)):
            if type(r) is not int or not 0 <= r < 48: return f'{at}: register {name}={r!r} is not an integer 0..47'
        a, b = R[i], R[j]
        if a is None: return f'{at}: reads register {i}, which holds no value yet'
        if b is None: return f'{at}: reads register {j}, which holds no value yet'
        if op == '-' and a <= b: return f'{at}: {a} - {b} is not positive'
        if op == '/' and a % b: return f'{at}: {a} / {b} is not exact'
        v = a + b if op == '+' else a - b if op == '-' else a * b if op == '*' else a // b
        if v > p['limit']: return f'{at}: value {v} is above value_limit {p["limit"]}'
        R[d] = v; seen.add(v)
    miss = [t for t in p['targets'] if t not in seen]
    if miss: return f'{len(miss)} of {len(p["targets"])} targets never appear in a register (first missing: {miss[0]})'
    return None

def check_file(path, col='program', strict=True):
    """Read an id,program CSV and check every row like the scorer (Evaluation page: exactly the columns id,program, every id
    once, at most 4,096 instructions and 128 KiB per cell, every program valid). Returns ({id: program or None}, [(id, problem)])."""
    rows, errs, seen = {}, [], Counter()
    with open(path, encoding='utf-8', newline='') as f:
        rd = csv.DictReader(f)
        if not rd.fieldnames or 'id' not in rd.fieldnames or col not in rd.fieldnames:
            return {}, [('(file)', f'needs the columns id,{col}; found {rd.fieldnames}')]
        if strict and rd.fieldnames != ['id', col]:
            errs.append(('(file)', f'the scorer wants exactly two columns id,{col} in that order; found {rd.fieldnames}'))
        for r in rd:
            k = r['id']; seen[k] += 1
            if k not in P: errs.append((k, 'unknown puzzle id')); continue
            if len(r[col].encode('utf-8')) > 131072: errs.append((k, 'program cell is larger than 131,072 bytes')); rows[k] = None; continue
            try: prog = json.loads(r[col])
            except Exception as e: errs.append((k, f'program is not valid JSON ({str(e)[:60]})')); continue
            e = explain(P[k], prog)
            if not e and len(prog) > 4096: e = f'{len(prog)} instructions, the limit is 4,096'
            if e: errs.append((k, e)); rows[k] = None
            else: rows[k] = prog
    errs += [(k, f'id appears {n} times (the scorer needs exactly one row per id)') for k, n in seen.items() if n > 1]
    errs += [(k, 'missing from the file') for k in sorted(P) if k not in seen]
    return rows, errs

# ---------- per-puzzle table ----------
def lclass(n): return '<=50' if n <= 50 else '51-100' if n <= 100 else '101-150' if n <= 150 else '151-200' if n <= 200 else '>200'
def tclass(n): return '1-9' if n < 10 else '10-19' if n < 20 else '20-29' if n < 30 else '30-31' if n < 32 else '32+'
def bclass(b): return '<=29' if b < 30 else '30-39' if b < 40 else '40-49' if b < 50 else '50-54' if b < 55 else '55-60'
LORD, TORD, BORD = ['<=50', '51-100', '101-150', '151-200', '>200'], ['1-9', '10-19', '20-29', '30-31', '32+'], ['<=29', '30-39', '40-49', '50-54', '55-60']
SORD = ['narrow', 'wide']   # narrow: every new target within 4 bits of the largest one (a documented rule, not the generator's label)

def floor_of(p):
    """Proven lower bound on the program length. (a) Every target that is not a start value is written by its own
    instruction. (b) Values grow at most by squaring: after j instructions no register exceeds B_j, with B_0 = max start
    and B_j = max(2, B_{j-1}^2) (a+b <= a*b once both are >= 2). A target above B_j cannot be written in the first j
    instructions, so if m targets are above B_j the program has at least j + m instructions. Floor = max over j."""
    st = set(p['start']); new = sorted(t for t in set(p['targets']) if t not in st)
    lo, B, j = len(new), max(p['start']), 0
    while new and B < new[-1]:
        j += 1; B = max(2, B * B)
        lo = max(lo, j + sum(1 for t in new if t > B))
    return lo, len(new)

SRC, FLOOR = {}, {}
def prepare():   # credits per puzzle and the floor of every puzzle (inside run(): a problem here cannot stop the notebook)
    SRC.update({r['id']: r['source'] for r in csv.DictReader(open(find('best_programs.csv')))})
    for k in CHANGE: SRC[k] = "this notebook's CHANGE cell"
    FLOOR.update({k: floor_of(P[k]) for k in P})
run(prepare)
def frame(other):
    """One row per puzzle: today's best length, the other file's length (NaN if invalid or missing) and the gap."""
    out = []
    for k in sorted(P):
        b = len(programs[k]); o = other.get(k); lo, nnew = FLOOR[k]
        src = SRC.get(k, ''); author = src.split(',')[0].strip() or '(unknown)'
        nb = [t.bit_length() for t in set(P[k]['targets']) if t not in set(P[k]['start'])] or [0]
        out.append(dict(id=k, best=b, other=len(o) if o is not None else np.nan, floor=lo, new_targets=nnew,
                        bits=max(P[k]['targets']).bit_length(), spread=max(nb) - min(nb), author=author,
                        notebook=src[len(author):].lstrip(', ')))
    df = pd.DataFrame(out); df['gap'] = df.other - df.best
    df['lost'] = df.gap.clip(lower=0); df['ahead'] = (-df.gap).clip(lower=0)
    df['L'] = df.best.map(lclass); df['T'] = df.new_targets.map(tclass); df['Bt'] = df.bits.map(bclass)
    df['S'] = np.where(df.spread <= 4, 'narrow', 'wide')
    return df

# ---------- cards ----------
def tiles(items, bg=NAVY):
    h = ''.join(f'<div style="display:inline-block;vertical-align:top;background:{bg};color:#ffffff;border-radius:10px;'
                f'padding:9px 16px;margin:6px 8px 0 0;min-width:130px;"><div style="font-size:24px;font-weight:800;color:#ffffff;">{a}</div>'
                f'<div style="font-size:12.5px;color:#e8edf7;">{b}</div></div>' for a, b in items)
    display(HTML(f'<div style="{FONT}">{h}</div>'))
def about(title, how, what, so):
    td = 'padding:6px 10px;border:1px solid #c9d2e0;vertical-align:top;color:#111111;'
    rows = ''.join(f'<tr><td style="{td}background:#dde6f3;font-weight:700;white-space:nowrap;">{a}</td><td style="{td}background:#ffffff;">{b}</td></tr>'
                   for a, b in (('How to read it', how), ('What it shows', what), ('So what', so)))
    display(HTML(f'<div style="{FONT}font-size:15px;line-height:1.55;overflow-x:auto;margin:6px 0 14px 0;"><div style="font-weight:700;'
                 f'color:#111111;background:#ffffff;padding:2px 4px;">About the figure: {title}</div><table style="border-collapse:collapse;'
                 f'background:#ffffff;">{rows}</table></div>'))
def table(df, cols, heads, title):
    th = 'padding:5px 9px;border:1px solid #c9d2e0;background:#14213d;color:#ffffff;text-align:left;'
    td = 'padding:4px 9px;border:1px solid #c9d2e0;background:#ffffff;color:#111111;'
    esc = lambda x: str(x).replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
    body = ''.join('<tr>' + ''.join(f'<td style="{td}">{esc(r[c])}</td>' for c in cols) + '</tr>' for _, r in df.iterrows())
    hh = ''.join(f'<th style="{th}">{esc(h)}</th>' for h in heads)
    display(HTML(f'<div style="{FONT}font-size:14px;overflow-x:auto;margin:6px 0 12px 0;"><div style="font-weight:700;color:#111111;'
                 f'background:#ffffff;padding:2px 4px;">{title}</div><table style="border-collapse:collapse;background:#ffffff;">'
                 f'<tr>{hh}</tr>{body}</table></div>'))

# ---------- the audit (Parts 1 and 2; Part 4 runs the same on your file) ----------
def concentration(df):
    g = np.sort(df.lost.values[df.lost.values > 0])[::-1]; tot = g.sum()
    if tot == 0: return g, 0, 0, 0
    cum = np.cumsum(g) / tot
    return g, tot, int(np.searchsorted(cum, .5) + 1), int(np.searchsorted(cum, .8) + 1)

def audit_numbers(df, name):
    v = df[df.other.notna()]; g, tot, n50, n80 = concentration(df)
    tiles([(f'{int(v.other.sum()):,}', f'{name}: instructions on {len(v):,} valid programs'),
           (f'{int(df.best.sum()):,}', 'today\'s best programs (this notebook)'),
           (f'{int(tot):,}', f'instructions lost on {int((df.lost > 0).sum()):,} puzzles'),
           (f'{n50} / {n80}', 'puzzles that hold 50% / 80% of the loss'),
           (f'{int(df.lost.max()) if tot else 0}', 'largest loss on one puzzle')])
    if (df.ahead > 0).any():
        tiles([(f'{int(df.ahead.sum()):,}', f'instructions where {name} is AHEAD, on {int((df.ahead > 0).sum())} puzzle(s)')], bg=GREEN)
    return tot, n50, n80

def fig_concentration(df, name):
    g, tot, n50, n80 = concentration(df)
    if tot == 0: print('no puzzle is behind: nothing to plot'); return
    fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={'width_ratios': [1.25, 1]})
    x = np.arange(1, len(g) + 1); cum = np.cumsum(g) / tot * 100
    ax[0].fill_between(x, cum, color=BLUE, alpha=.15); ax[0].plot(x, cum, color=BLUE, lw=2.2)
    for n, q, c in ((n50, 50, RED), (n80, 80, GOLD)):
        ax[0].plot([n, n, 0], [0, q, q], color=c, lw=1.4, ls='--')
        ax[0].annotate(f'{n} puzzles hold {q}%', (n, q), xytext=(n + len(g) * .04, q - 12), color=c, fontweight='bold')
    ax[0].set(xlim=(0, len(g) * 1.02), ylim=(0, 102), xlabel='puzzles behind, largest loss first', ylabel='% of the instructions lost',
              title='How concentrated the loss is')
    ax[0].grid(alpha=.35, color=GRID)
    edges = [(1, 1, '1'), (2, 2, '2'), (3, 4, '3-4'), (5, 9, '5-9'), (10, 10 ** 9, '10+')]
    lab = [e[2] for e in edges]; ins = [int(g[(g >= a) & (g <= b)].sum()) for a, b, _ in edges]
    cnt = [int(((g >= a) & (g <= b)).sum()) for a, b, _ in edges]
    bars = ax[1].bar(lab, ins, color=[GREY, GREY, BLUE, NAVY, RED])
    for bb, i, c in zip(bars, ins, cnt):
        ax[1].text(bb.get_x() + bb.get_width() / 2, i, f'{i:,} instr.\n{c} puzzles', ha='center', va='bottom', fontsize=10)
    ax[1].set(xlabel='instructions lost on one puzzle', ylabel='instructions lost in total', title='Small losses vs big losses',
              ylim=(0, max(ins) * 1.3))
    plt.tight_layout(); plt.show()
    big = sum(i for i, (a, b, _) in zip(ins, edges) if a >= 5)
    about('concentration', 'Left: puzzles sorted from the largest loss to the smallest; the curve adds up their share of the total. '
          'Right: the same losses grouped by size; the label says how many instructions and puzzles each group holds.',
          f'{name[0].upper() + name[1:]} is behind on {len(g)} puzzles by {int(tot):,} instructions. {n50} puzzles ({n50 / len(P):.0%} of all 1,000) hold half '
          f'of it, {n80} hold 80%. Losses of 5 or more on one puzzle sum to {big:,} instructions ({big / tot:.0%}); one-instruction '
          f'losses on {cnt[0]} puzzles sum to {ins[0]:,} ({ins[0] / tot:.0%}).',
          'Every lost instruction is in best_programs.csv (or this notebook\'s CHANGE cell), so a per-puzzle merge closes all of it. '
          'The curve says how much of that comes from a few puzzles, and Part 2 says which kind of puzzle your own method misses.')

def fig_where(df, name):
    fig, axs = plt.subplots(1, 4, figsize=(15, 4.3), gridspec_kw={'width_ratios': [5, 5, 5, 2.4]})
    tot = df.lost.sum() or 1
    for ax, col, order, lab, c in ((axs[0], 'L', LORD, "today's best program length", BLUE),
                                   (axs[1], 'T', TORD, 'number of targets to build', PURPLE),
                                   (axs[2], 'Bt', BORD, 'bits of the largest target', RED),
                                   (axs[3], 'S', SORD, 'target sizes', GOLD)):
        grp = df.groupby(col); rate = [100 * grp.lost.sum().get(o, 0) / max(grp.best.sum().get(o, 1), 1) for o in order]
        share = [grp.lost.sum().get(o, 0) / tot for o in order]; n = [int(grp.size().get(o, 0)) for o in order]
        bars = ax.bar(order, rate, color=c, alpha=.9)
        for bb, r, s, k in zip(bars, rate, share, n):
            ax.text(bb.get_x() + bb.get_width() / 2, r, f'{s:.0%}\n({k})', ha='center', va='bottom', fontsize=9.5)
        ax.set(xlabel=lab, ylim=(0, max(rate + [0.1]) * 1.35)); ax.grid(axis='y', alpha=.35, color=GRID)
        ax.tick_params(axis='x', labelsize=9.5)
    axs[0].set_ylabel('instructions lost per 100 of the best')
    axs[0].set_title('Where the loss sits'); plt.tight_layout(); plt.show()
    grp = df.groupby('L'); top = max(LORD, key=lambda o: grp.lost.sum().get(o, 0))
    gb = df.groupby('Bt'); topb = max(BORD, key=lambda o: 100 * gb.lost.sum().get(o, 0) / max(gb.best.sum().get(o, 1), 1))
    about('where the loss sits', 'Bar height = instructions lost per 100 instructions of today\'s best programs in that group '
          '(a rate, so groups of different size compare fairly). Label = share of the total loss, and (puzzles in the group). '
          'Last panel: <i>narrow</i> = every target within 4 bits of the largest one, <i>wide</i> = sizes spread over more bits.',
          f'The largest share of {name}\'s loss is in programs of length {top} ({grp.lost.sum()[top] / tot:.0%}); the highest '
          f'rate by target size is in the {topb}-bit group ({100 * gb.lost.sum()[topb] / gb.best.sum()[topb]:.2f} per 100).',
          'The share says where most of the lost instructions are; the rate says where the file is relatively weakest, '
          'even if that group is small. A method gap usually shows up as a high rate in one group, not as an even spread.')

def fig_who(df, name):
    d = df[df.lost > 0]
    if d.empty: return
    w = d.groupby('author').lost.agg(['sum', 'size']).sort_values('sum', ascending=False)
    top = w.head(10)[::-1]; tot = d.lost.sum()
    fig, ax = plt.subplots(figsize=(10.5, 0.42 * len(top) + 1.2))
    ax.barh(top.index, top['sum'], color=GREEN)
    for i, (s, n) in enumerate(zip(top['sum'], top['size'])): ax.text(s, i, f'  {int(s):,} on {int(n)} puzzles', va='center', fontsize=10)
    ax.set(xlabel=f'instructions lost by {name}, credited to the author of today\'s program', xlim=(0, top['sum'].max() * 1.35),
           title='Whose program closed the gap')
    plt.tight_layout(); plt.show()
    a0 = w.index[0]
    about('whose program closed the gap', 'For every puzzle where the other file is longer, the loss is credited to the author '
          'of today\'s best program for that puzzle (column source of best_programs.csv).',
          f'{a0}\'s programs close {int(w["sum"].iloc[0]):,} of the {int(tot):,} instructions ({w["sum"].iloc[0] / tot:.0%}); '
          f'{len(w)} authors appear in total.',
          'Their notebooks are the first to study: the method behind the biggest bar is what closed most of this gap.')
    t = d.sort_values(['lost', 'best'], ascending=[False, False]).head(12).copy()
    t['other'] = t.other.astype(int); t['lost'] = t.lost.astype(int)
    table(t, ['id', 'other', 'best', 'lost', 'new_targets', 'bits', 'author'],
          ['puzzle', name, 'best', 'lost', 'targets', 'bits', "today's program by"], 'The 12 largest single-puzzle losses')

def audit(other, name):
    df = frame(other); run(audit_numbers, df, name); run(fig_concentration, df, name)
    return df


In [ ]:
# Part 1: what changed in the public frontier since the day-old file (both replayed under the rules).
def fam(r):   # family = target-size rule (narrow: every new target within 4 bits of the largest) + bits of the largest target
    return f"{r['S']}, {r['bits']}-bit"
def short(s, n=44): return s if len(s) <= n else s[:n - 1] + '…'
def fl(s, n=28):   # figure label: the default plot font has no CJK glyphs, so those characters show as '?'
    return short(s, n) if all(ord(ch) < 0x250 for ch in s) else '(non-Latin name, see table)'

def part1():
    global DAY, DAYUTC, DAYTOT
    path = find('public_day_ago.csv'); rows = list(csv.DictReader(open(path)))
    day, errs = check_file(path, strict=False)   # this file carries two extra columns (utc, total)
    if errs: print(f'{len(errs)} program(s) of the day-old file do not pass the checker, e.g.', errs[:3])
    DAYUTC, DAYTOT = rows[0]['utc'], int(rows[0]['total'])
    DAY = df = frame(day)
    lin = pd.read_csv(find('public_lineage.csv')); lin['score'] = lin.score.astype(int); lin['work'] = lin.work.astype(str); lin['author'] = lin.author.astype(str)
    lin = lin.sort_values('utc', kind='stable'); end = lin.utc.iloc[-1]
    hours = (pd.Timestamp(end) - pd.Timestamp(DAYUTC)).total_seconds() / 3600
    imp = df[df.gap > 0].copy(); imp['gain'] = imp.gap.astype(int); imp['old'] = imp.other.astype(int)
    today = int(df.best.sum()); before = int(df.other.sum())
    w = imp.groupby('author').gain.agg(['sum', 'size']).sort_values(['sum', 'size'], ascending=False)
    win = lin[(lin.utc > DAYUTC)]; rec, best = [], DAYTOT
    for _, r in win.iterrows():
        if r.score < best: rec.append(dict(utc=r.utc, score=r.score, step=best - r.score, author=r.author, work=r.work)); best = r.score
    rec = pd.DataFrame(rec)
    tiles([(f'{before - today:,}', f'instructions saved in the last {hours:.0f} h'),
           (f'{len(imp):,}', 'puzzles improved'),
           (w.index[0] if len(w) else '-', f'top contributor: {int(w["sum"].iloc[0]) if len(w) else 0:,} instr. on {int(w["size"].iloc[0]) if len(w) else 0} puzzles'),
           (f'{len(rec):,}', 'new best public scores'), (f'{today:,}', 'best public total now')])
    lastsc = int(lin.score.iloc[-1])
    more = '' if lastsc == today else (f' The dataset holds programs newer than the last release in the score list ({lastsc:,} at {end} UTC), '
                                       f'so the figures below include them while the list of new best scores stops at that release.')
    display(HTML(f'<div style="{FONT}font-size:15.5px;line-height:1.6;color:#111111;background:#f3f6fc;border-left:6px solid {BLUE};'
                 f'border-radius:8px;padding:10px 16px;margin:8px 0;">Window: from the best public file at <b>{DAYUTC} UTC</b> '
                 f'(<b>{DAYTOT:,}</b> instructions; replayed here: {before:,}) to the best public programs in this dataset '
                 f'(<b>{today:,}</b> replayed here). The difference is the sum of the one-puzzle improvements below.{more}</div>'))
    if imp.empty: print('no puzzle changed in the window'); return
    # --- figure 1: who, and which kind of program
    g = imp.sort_values('gain', ascending=False)
    fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.6), gridspec_kw={'width_ratios': [1.35, 1]})
    top = w.head(8)[::-1]
    ax[0].barh([fl(a) for a in top.index], top['sum'], color=GREEN)
    for i, (s, n) in enumerate(zip(top['sum'], top['size'])): ax[0].text(s, i, f'  {int(s):,} on {int(n)} puzzles', va='center', fontsize=10)
    ax[0].set(xlim=(0, top['sum'].max() * 1.4), xlabel='instructions saved, credited to the author of the current program',
              title='Gain by contributor')
    rate = [100 * imp[imp.L == o].gain.sum() / max(df[df.L == o].best.sum(), 1) for o in LORD]
    sh = [imp[imp.L == o].gain.sum() / max(imp.gain.sum(), 1) for o in LORD]; n = [int((imp.L == o).sum()) for o in LORD]
    bars = ax[1].bar(LORD, rate, color=BLUE)
    for bb, r, s, k in zip(bars, rate, sh, n): ax[1].text(bb.get_x() + bb.get_width() / 2, r, f'{s:.0%}\n({k})', ha='center', va='bottom', fontsize=9.5)
    ax[1].set(xlabel='length of the current program', ylabel='instructions saved per 100', ylim=(0, max(rate + [.1]) * 1.35),
              title='Gain by program length'); ax[1].grid(axis='y', alpha=.35, color=GRID)
    plt.tight_layout(); plt.show()
    a0 = w.index[0]; topL = LORD[int(np.argmax(rate))]
    about('who gained, and on what', 'Left: instructions saved in the window, summed per author of the program now in best_programs.csv '
          '(label: instructions and number of puzzles). Right: the same gain grouped by the length of the current program; bar = saved '
          'per 100 instructions of that group, label = share of all gain and (puzzles improved).',
          f'{len(imp)} of 1,000 puzzles got shorter, {before - today:,} instructions in total; the median gain is {int(imp.gain.median())}, '
          f'the largest {int(imp.gain.max())}. {a0} accounts for {w["sum"].iloc[0] / imp.gain.sum():.0%}; {len(w)} authors appear. '
          f'The saving rate is highest for programs of length {topL}.',
          'Credit goes to whoever wrote the program that is now the best for the puzzle; a file that merged someone else\'s programs '
          'is not credited for them. Check the notebook of the top bar first.')
    # --- table: every improved puzzle (top 30)
    t = g.head(30).copy(); t['family'] = t.apply(fam, axis=1); t['move'] = t.old.astype(str) + ' → ' + t.best.astype(str)
    t['by'] = t.author + ': ' + t.notebook.map(short); t['gain'] = '-' + t.gain.astype(str)
    rest = len(g) - len(t)
    table(t, ['id', 'family', 'move', 'gain', 'by'], ['puzzle', 'family', 'old → new', 'gain', 'program by (author: notebook)'],
          f'The {len(t)} largest improvements' + (f' (+ {rest} more puzzles, {int(g.gain.iloc[len(t):].sum())} instructions)' if rest else ''))
    display(HTML(f'<div style="{FONT}font-size:13.5px;color:#111111;background:#ffffff;padding:2px 4px;">Family = the target-size rule of '
                 f'Part 2 (<i>narrow</i>: every new target within 4 bits of the largest; <i>wide</i>: spread over more bits) and the bits of '
                 f'the largest target.</div>'))
    # --- the day's new best scores
    if len(rec):
        fig, ax = plt.subplots(figsize=(12.5, 4.0))
        xs = [pd.Timestamp(DAYUTC)] + [pd.Timestamp(u) for u in rec.utc]; ys = [DAYTOT] + list(rec.score)
        ax.step(xs, ys, where='post', color=NAVY, lw=1.8)
        names = list(rec.author.value_counts().index); pal = [BLUE, RED, GREEN, GOLD, PURPLE, GREY]
        for k, a in enumerate(names[:5] + ['others']):
            m = rec.author.isin(names[:5]) & (rec.author == a) if a != 'others' else ~rec.author.isin(names[:5])
            if m.any(): ax.scatter([pd.Timestamp(u) for u in rec.utc[m]], rec.score[m], s=46, color=pal[k], zorder=3, label=f'{fl(a, 26)} ({int(m.sum())})')
        ax.set(ylabel='best public score (instructions)', title=f'The {len(rec)} new best public scores'); ax.grid(alpha=.35, color=GRID)
        ax.legend(loc='upper right', fontsize=9.5, ncol=2); fig.autofmt_xdate(); plt.tight_layout(); plt.show()
        about('new best scores', 'Each dot is a public release that beat every earlier public score in the window; the line is the best public '
              'score at that time (UTC), starting from the day-old file.',
              f'{len(rec)} new records, from {DAYTOT:,} to {int(rec.score.iloc[-1]):,}; the largest single step was {int(rec.step.max()):,} '
              f'({rec.loc[rec.step.idxmax(), "author"]}). {names[0]} set {int(rec.author.value_counts().iloc[0])} of them.',
              'Small steps late in the window are re-runs of the same optimizers on a few puzzles; big steps are new methods or merges.')
        r = rec.tail(15).copy(); r['score'] = r.score.map('{:,}'.format); r['step'] = '-' + r.step.astype(str); r['work'] = r.work.map(lambda s: short(s, 60))
        table(r, ['utc', 'score', 'step', 'author', 'work'], ['UTC', 'score', 'step', 'author', 'notebook'],
              f'The latest {len(r)} of {len(rec)} new best public scores' if len(rec) > len(r) else 'The new best public scores')
run(part1)

<a id="p2"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#6a4c93,#3d2a66);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 2</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">Where the gap sits</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">By program length, number of targets and target size, on the same day-old comparison</div></div>

In [ ]:
# Part 2: in which kind of puzzle the day-old file loses, on the same comparison (who closed it is Part 1).
if 'DAY' in globals():
    run(fig_concentration, DAY, 'the day-old file'); run(fig_where, DAY, 'the day-old file')

<a id="p3"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#0f9a68,#0a6a49);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 3</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">Where everyone still loses</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">A proven lower bound per puzzle, and what today's best programs spend above it</div></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#eaf6f0;border-left:6px solid #0f9a68;border-radius:8px;padding:12px 18px;margin:10px 0;"><b>The floor in one paragraph.</b> An instruction writes one value, so every target that is not a start value costs at least one instruction. Values also cannot grow faster than squaring, so a target of 50 bits built from start values of 10 bits needs a few instructions before it can exist at all. The notebook takes the larger of these counts per puzzle (function <code style="background:#e3e8f2;color:#111111;padding:1px 6px;border-radius:4px;">floor_of</code>). It is <b>proven</b> (no program can be shorter) but <b>weak</b>: it ignores how hard it is to hit a large number exactly. It tells you which instructions are unavoidable, not how short a program can really be.</div>

In [ ]:
# Part 3: the proven floor per puzzle (see floor_of in the toolkit) and what today's best programs spend above it.
def part3():
    f = frame({}); f['above'] = f.best - f.floor; f['per'] = f.above / f.new_targets.clip(lower=1)
    lo, be = int(f.floor.sum()), int(f.best.sum())
    tiles([(f'{lo:,}', 'proven floor, all 1,000 puzzles'), (f'{be:,}', "today's best programs"),
           (f'{be - lo:,}', f'instructions above the floor ({(be - lo) / be:.0%} of all)'),
           (f'{int((f.above == 0).sum())}', 'puzzles where the best already equals the floor')], bg=PURPLE)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={'width_ratios': [1, 1.2]})
    g = f.groupby('Bt'); nt = g.new_targets.sum().reindex(BORD)
    fl = (g.floor.sum().reindex(BORD) / nt).values; ab = (g.above.sum().reindex(BORD) / nt).values
    ax[0].bar(BORD, fl, color=GREY, label='floor'); ax[0].bar(BORD, ab, bottom=fl, color=PURPLE, label='above the floor')
    for i, (a, b) in enumerate(zip(fl, ab)): ax[0].text(i, a + b, f'{a + b:.2f}', ha='center', va='bottom', fontsize=10)
    ax[0].set(xlabel='bits of the largest target', ylabel='instructions per target', title='Floor vs what the best spends',
              ylim=(0, np.nanmax(fl + ab) * 1.2)); ax[0].legend(loc='upper left'); ax[0].grid(axis='y', alpha=.35, color=GRID)
    rng = np.random.default_rng(0)
    for o, c, lab in (('wide', BLUE, 'wide: target sizes spread over more than 4 bits'), ('narrow', RED, 'narrow: all targets within 4 bits of the largest')):
        q = f[f.S == o]; ax[1].scatter(q.bits + rng.uniform(-.35, .35, len(q)), q.per, s=9, alpha=.55, color=c, label=f'{lab} ({len(q)})')
    ax[1].set(xlabel='bits of the largest target (jittered)', ylabel='instructions above the floor per target',
              title='Every puzzle: spend above the floor'); ax[1].legend(loc='upper left', markerscale=2)
    ax[1].grid(alpha=.35, color=GRID); plt.tight_layout(); plt.show()
    big = f[f.bits >= 40]; per = lambda d: d.above.sum() / max(d.new_targets.sum(), 1)
    nw, wd = big[big.S == 'narrow'], big[big.S == 'wide']
    about('the floor', 'Left: grey = the proven floor per target (about 1: every new target needs its own instruction, plus the few '
          'squaring steps before a large target can exist); purple = what today\'s best programs spend above it, by size of the '
          'largest target. Right: one dot per puzzle, split by a simple rule on the targets of the puzzle (narrow or wide sizes).',
          f'The best programs spend {be / lo:.1f}x the floor; {be - lo:,} of the {be:,} instructions are helpers (values that are not '
          f'targets). Spend above the floor grows with target size. Among puzzles with a largest target of 40+ bits, the '
          f'{len(nw)} <b>narrow</b> ones spend {per(nw):.2f} instructions per target above the floor and the {len(wd)} <b>wide</b> '
          f'ones {per(wd):.2f}: similar target sizes, {per(nw) / max(per(wd), 1e-9):.1f}x the cost.',
          'The floor is a bound, not a target: nobody knows how close to it a program can get. It does show where every future '
          'gain must come from, for you and for the leaders: fewer helper instructions. Narrow puzzles are where the most helpers '
          'are spent per target. A likely reason (not tested here): in a wide puzzle the smaller targets can serve as helpers for '
          'the larger ones, while a narrow puzzle has only equally large targets and a few small start values to build from.')
    t = f.sort_values('per', ascending=False).head(8).copy(); t['per'] = t.per.round(2)
    table(t, ['id', 'best', 'floor', 'per', 'new_targets', 'bits', 'S', 'author'],
          ['puzzle', 'best', 'floor', 'above floor per target', 'targets', 'bits', 'sizes', 'best program by'],
          'The 8 puzzles that spend the most above the floor per target')
run(part3)

<a id="p4"></a>
<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;background:linear-gradient(120deg,#b22222,#7a1717);color:#ffffff;padding:18px 24px;border-radius:12px;margin:22px 0 8px 0;"><div style="font-size:12.5px;letter-spacing:2px;text-transform:uppercase;color:#ffffff;">Part 4</div><div style="font-size:26px;font-weight:800;line-height:1.2;margin-top:4px;color:#ffffff;">Your submission</div><div style="font-size:15.5px;margin-top:6px;color:#ffffff;">Set MY_SUBMISSION and the same audit runs on your file, with a strict checker</div></div>

<div style="font-family:'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:15.5px;line-height:1.6;color:#111111;background:#fdeeee;border-left:6px solid #b22222;border-radius:8px;padding:12px 18px;margin:10px 0;"><b>How to use it.</b> Copy &amp; Edit, add your own output or dataset as an input, set <code style="background:#e3e8f2;color:#111111;padding:1px 6px;border-radius:4px;">MY_SUBMISSION = '/kaggle/input/.../submission.csv'</code> in the next cell and run all. You get: the exact problem of every invalid program (instruction number, the instruction, the rule), your total, the concentration and the where/who figures of Parts 1-2 for your file, and the puzzles where you are <b>ahead</b> of the best public programs. Any <code style="background:#e3e8f2;color:#111111;padding:1px 6px;border-radius:4px;">id,program</code> CSV works, also another notebook's output.</div>

In [ ]:
# Part 4: your submission. Attach it as an input (Add Input -> your notebook output or dataset) and set the path.
MY_SUBMISSION = None   # e.g. '/kaggle/input/my-santa-run/submission.csv'

def part4():
    if MY_SUBMISSION:
        mine, errs = check_file(MY_SUBMISSION)
        ok = sum(1 for v in mine.values() if v)
        print(f'{MY_SUBMISSION}: {ok} of {len(P)} programs valid, {len(errs)} problem(s)')
        for k, e in errs[:20]: print(f'  INVALID {k}: {e}')
        if errs: print('  (the scorer rejects a file with any invalid program; the audit below uses the valid ones only)')
        df = audit({k: v for k, v in mine.items() if v}, 'your file')
        run(fig_where, df, 'your file'); run(fig_who, df, 'your file')
        if (df.ahead > 0).any():
            t = df[df.ahead > 0].sort_values('ahead', ascending=False).head(12).copy(); t['other'] = t.other.astype(int)
            table(t, ['id', 'other', 'best', 'ahead', 'author'], ['puzzle', 'yours', 'best public', 'you save', 'best public by'],
                  'Where you are ahead of the best public programs (consider sharing them, or keep them)')
        return
    # No file set: show what the checker says on broken copies of a real program.
    k0 = max(P, key=lambda k: len(programs[k])); base = [list(x) for x in programs[k0]]
    def mut(f):
        q = [list(x) for x in base]; f(q); return q
    def swap_sub(q):
        i = next((i for i, x in enumerate(q) if x[0] == '-'), None)
        if i is not None: q[i][1], q[i][2] = q[i][2], q[i][1]
    cases = [('the real program', base),
             ('first instruction reads register 47', mut(lambda q: q[0].__setitem__(1, 47))),
             ("operator '+-'", mut(lambda q: q[0].__setitem__(0, '+-'))),
             ('destination given as "4" (a string)', mut(lambda q: q[0].__setitem__(3, '4'))),
             ('3 fields instead of 4', mut(lambda q: q[0].pop())),
             ('first subtraction turned around', mut(swap_sub)),
             ('last instruction dropped', base[:-1])]
    t = pd.DataFrame([dict(case=c, msg=explain(P[k0], q) or 'valid') for c, q in cases])
    table(t, ['case', 'msg'], ['broken copy of ' + k0, 'what the checker says'],
          'MY_SUBMISSION is not set: the checker on broken copies of a real program')
    import tempfile                     # and the file-level checks, on a 4-row CSV written to a temporary folder
    k1, k2 = sorted(P)[:2]; demo = os.path.join(tempfile.mkdtemp(), 'demo.csv')
    with open(demo, 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(['id', 'program'])
        for k, v in ((k1, json.dumps(programs[k1])), (k1, json.dumps(programs[k1])), (k2, '[["+",0,1,'), ('scored_9999', '[]')): w.writerow([k, v])
    _, errs = check_file(demo)
    t = pd.DataFrame([dict(id=k, msg=e) for k, e in errs[:4]] + [dict(id='...', msg=f'{len(errs) - 4} more rows: missing from the file')])
    table(t, ['id', 'msg'], ['id', 'what the checker says'], 'File-level checks on a 4-row demo file')
run(part4)
print(f'analysis time: {time.time() - T0:.1f} s')

## Write

In [ ]:
if WRITE_SUBMISSION:
    with open('/kaggle/working/submission.csv', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['id', 'program'])
        for pid in sorted(programs): w.writerow([pid, json.dumps(programs[pid], separators=(',', ':'))])
    print(f'submission.csv written: {TOTAL:,} instructions')
else:
    print('WRITE_SUBMISSION is False: no submission file in this output. Copy & Edit and set it to True.')